# 05 - Market Share Analysis

Purpose:
Show how HealthSynth generates baseline and adjusted market share over time.

Important note:
In this version, market share is generated as a separate analytical table. Prescription generation does not yet consume adjusted market share directly.

In [ ]:
from healthsynth.generator import generate
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
datasets = generate(
    config_path="../configs/profiles/oncology_training.yaml",
    output_dir="../output/market_share_analysis",
)

In [ ]:
products = datasets["product"]
market_share = datasets["market_share"]

products

In [ ]:
market_share.head()

In [ ]:
market_share = market_share.merge(
    products[["product_id", "product_name", "manufacturer", "brand_type"]],
    on="product_id",
)
market_share["month"] = pd.to_datetime(market_share["month"])
market_share.head()

In [ ]:
baseline = (
    products[["product_name", "baseline_market_share"]]
    .sort_values("baseline_market_share", ascending=False)
)

plt.figure(figsize=(8, 5))
plt.bar(baseline["product_name"], baseline["baseline_market_share"])
plt.title("Baseline Market Share by Product")
plt.xlabel("Product")
plt.ylabel("Baseline Market Share")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
pivot = market_share.pivot_table(
    index="month",
    columns="product_name",
    values="adjusted_market_share",
)

pivot.head()

In [ ]:
plt.figure(figsize=(10, 5))

for column in pivot.columns:
    plt.plot(pivot.index, pivot[column], label=column)

plt.title("Adjusted Market Share Over Time")
plt.xlabel("Month")
plt.ylabel("Adjusted Market Share")
plt.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
share_check = (
    market_share.groupby(["therapeutic_area", "month"])["adjusted_market_share"]
    .sum()
    .reset_index(name="total_share")
)

share_check.head()

In [ ]:
rx = datasets["prescriptions"].merge(
    products[["product_id", "product_name", "manufacturer", "brand_type"]],
    on="product_id",
)

rx_by_product = (
    rx.groupby("product_name")["nrx"]
    .sum()
    .reset_index()
    .sort_values("nrx", ascending=False)
)

rx_by_product

In [ ]:
plt.figure(figsize=(8, 5))
plt.bar(rx_by_product["product_name"], rx_by_product["nrx"])
plt.title("Generated NRx by Product")
plt.xlabel("Product")
plt.ylabel("Total NRx")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Key Takeaway

HealthSynth now generates a market_share table that captures baseline and adjusted product share over time.

This table is useful for learning:

- baseline share assumptions
- product competition
- month-by-month market movement
- how market share can be analyzed separately from prescription volume

In the current version, prescriptions are still generated independently from adjusted market share. A future version will use market share directly in prescription allocation.